# C1.4 · Establishing telemetry and detecting the actor

**Function C — Agentic Evaluation and Red Teaming → One Red-Team Lifecycle, End to End**

Builds on **[C1.3 · Cognitive vulnerability and elicitation scaling](https://spbreed.github.io/cyber-commons/lessons/C1.3.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry, OpenSearch |

## What this lesson is

**What it covers.** JSON-wrapping the model-gateway trace so an agent is observable at all, then scoring actors on behaviour to tell agent tool calls from human activity.

**Why a security engineer needs it.** This is the function's pivot: the offensive finding becomes defensive telemetry. The agents you most need to find are the ones in no registry, and the gateway logs none of the fields that would reveal them until you add them.

| | |
|---|---|
| **Day 0 — why** | To catch the actor you have just played, you have to be able to see it — and the model gateway logs none of the fields that would distinguish an agent from a person. |
| **Day 1 — how** | Wrap the gateway trace with identity, tool and arguments, then score actors on behaviour rather than on what they claim to be. |
| **Day 2 — measure** | Threshold chosen by expected cost rather than accuracy, and the unregistered actors it surfaces. |

## 1 · The hook

To catch the actor you just played you have to see it, and an agent's tool calls arrive through a gateway that logs none of the fields telling an agent from a person. This is where the offensive finding becomes defensive telemetry.

> **At CyberTravels.** The gateway is CyberTravels', and the actor to find is the Workflow Agent acting under Alex's credential at a tempo no person types — invisible until the trace carries the acting identity.

## 2 · The framework

```
   the model gateway, by default        wrapped for telemetry

   { } no per-call identity        ->    {"id": "svc-idx", "tool": "...",
   { } no tool arguments                  "args": {...}, "ts": ...}
   { } no acting principal                        |
                                                  v
                                    score actors on behaviour:
                                    regularity, rate, continuity
                                    threshold set by COST, not accuracy
```

The pivot of the whole function: an offensive finding becomes defensive
telemetry. To catch the actor you have just played, you have to be able to
**see it at all** — and an agent's tool calls arrive through a model gateway
that, by default, logs none of the fields that would tell an agent apart from a
person.

Establishing telemetry means JSON-wrapping the gateway trace — prompt, tool,
arguments, identity — and then scoring actors on behaviour, because the ones you
most need to find are the ones in no registry, acting under a human's
credential at a machine's tempo.

## 3 · The procedure, as a skill

The skill scores five CyberTravels actors on behaviour rather than on what they claim to be, and picks the threshold by expected cost — a flagged human costs half an analyst-hour, a missed agent costs forty.

### The skill — [`skills/detection/agent-versus-human-scoring/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-versus-human-scoring/SKILL.md)

```yaml
name: agent-versus-human-scoring
description: >-
  Score actors on behavioural signals to separate agents from people, sweep the
  threshold, and pick it by expected cost rather than by accuracy. Use when
  deciding whether a session is automated, or when unregistered automation needs
  finding.
allowed-tools: Read, Grep, Glob
```

# Pick the threshold by what each mistake costs

Separating agent from human is a scoring problem with two asymmetric errors: a
flagged human costs an analyst half an hour, and a missed agent costs whatever
an unmonitored automation does. Choosing the threshold by accuracy weights those
equally, which is the one thing you know is wrong.

## When to use this

Finding unregistered automation, deciding whether a session is a person, and
before any control that treats agents differently from users.

## Procedure

**1 — Score on behaviour, not on the user agent string.** Inter-action variance,
rate, breadth, and the share of actions with no preceding read. Anything
self-declared is a claim.

**2 — Score a spread of real actors.** A service indexer, an unknown token, a
person, a person driving an IDE assistant, and an agent deliberately jittered to
look human. The last two are the interesting middle.

**3 — Sweep the threshold and record both errors.** Humans flagged and agents
missed, at each setting. They move in opposite directions and the crossing point
is not the answer.

**4 — Attach a cost to each error and minimise the total.** Analyst hours for a
false positive, expected hours of an unmonitored agent for a false negative. The
chosen threshold now has a justification somebody can argue with.

**5 — Join to the registry.** An actor scoring as an agent and absent from the
registry is the finding worth routing; a registered agent scoring as an agent is
working correctly.

## Example

**Input** — the fixture committed at the top of [`scripts/agent_versus_human_scoring.py`](scripts/agent_versus_human_scoring.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
actor                   score     cv   rate/s   span_h  truth
--------------------------------------------------------------
svc-indexer             0.800    0.0    20.04     0.01  agent
dana@corp               0.028   1.55      0.0     1.11  human
unknown-token-7f3c      0.563    0.0      1.0     0.11  agent
sam@corp-ide            0.533    0.0      0.5      0.1  human
polite-agent            0.021   1.26      0.0     0.83  agent
 threshold  humans flagged   agents MISSED
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "actors": [{"name": "str", "score": 0.0, "truth": "agent|human|unknown"}],
  "sweep": [{"threshold": 0.0, "humans_flagged": 0, "agents_missed": 0, "expected_cost": 0.0}],
  "costs": {"false_positive_hours": 0.0, "false_negative_hours": 0.0},
  "chosen": {"threshold": 0.0, "why": "str"},
  "registry": {"scored_agent_unregistered": ["str"]}
}
```

## Failure modes

- **Scoring the user agent string.** It is self-declared.
- **Optimising accuracy.** It assumes the two errors cost the same.
- **Flagging registered agents.** They are supposed to look like agents.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/agent-versus-human-scoring/scripts/agent_versus_human_scoring.py
SCRIPT = "skills/detection/agent-versus-human-scoring/scripts/agent_versus_human_scoring.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The service accounts and unregistered token score highest, the human lowest, and cost-weighting selects a low threshold that finds the shadow agents at the price of a few analyst-hours.

## Your turn

Check whether your model gateway logs the acting identity per call. If it logs only the API key, every agent is anonymous and this scoring is the only actor you have.

---

**Next → [C1.5 · Emergent swarms and multi-agent proliferation](https://spbreed.github.io/cyber-commons/lessons/C1.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*